# LLM 12일차 (1) — PydanticAI `Agent` 로 LLM 출력 구조화 : `output_type` · 자동 검사 · 재시도

교재 `AI Agent` **p.139~144** (PydanticAI Agent 를 이용한 LLM 출력 구조화 — 자재 재고 분석 · 자동 발주 에이전트 · 실패 / 성공 시나리오 · 핵심 로직) 와 수업 필기 `pydantic_agent` 를 정리한 노트입니다.

```python
class InventoryAnalysis(BaseModel):                       # LLM 이 반드시 지켜야 할 답 형식
    item_name: str = Field(description='자재 품목 명칭')
    current_stock: int = Field(description='현재 재고 수량 (숫자만)')

pydantic_agent = Agent('openai:gpt-5-nano', output_type=InventoryAnalysis, system_prompt='…')
result = await pydantic_agent.run('…재고는 현재 15개 남음…')
result.output          # InventoryAnalysis(item_name='고강도 볼트M8', current_stock=15)  ← int 가 보장됨
```

- 서울시 상권 데이터 파이프라인 · 분석 에이전트는 [`seoul_commercial_db_pipeline.ipynb`](./seoul_commercial_db_pipeline.ipynb) · [`seoul_commercial_analysis_agent.ipynb`](./seoul_commercial_analysis_agent.ipynb)
- 원본에서 고친 부분 · 보충은 **📝** 로 적었습니다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

> **출력 읽는 법** — `# 📼` 수업 출력 옮김 · `# ⏳` 키가 필요해 비어 있는 셀 · `# 🧪` 가짜 모델(`FunctionModel` · `TestModel`)로 실제 실행 · 표시 없음 = 실제 실행

## 0. 설치 · 실행 방법

| 패키지 | 쓰는 것 | 설치 |
|---|---|---|
| `pydantic-ai` (또는 `pydantic-ai-slim[openai]`) | `Agent` · `FunctionModel` · `TestModel` | `pip install "pydantic-ai-slim[openai]"` |
| `langgraph` | `StateGraph` | Day 051~ |

`.env` 에 `OPENAI_API_KEY`. 키 없이도 `# 🧪` 셀은 실행된다 (정리 환경 : `pydantic-ai-slim 2.76.0` · `pydantic 2.14.3` · `langgraph 1.2.21`)

> **📝 중요 — `gpt-5-nano` 지원 종료 예정** 수업 출력(아래 📼)에 PydanticAI 가 낸 경고가 있다 : *Model 'openai:gpt-5-nano' is deprecated … retirement date of 2026-12-11. It recommends 'gpt-6-luna'*. OpenAI 지원 종료 목록에서도 **gpt-5-nano 는 2026-12-11 종료**로 확인했다. 이 과정의 모든 노트북이 `gpt-5-nano` 를 쓰므로 그날 이후에는 모델 이름을 바꿔야 실행된다.

In [1]:
# ① 설치 (처음 한 번)
# !pip install -q "pydantic-ai-slim[openai]"

In [2]:
import os, json, asyncio
os.environ.setdefault('PYDANTIC_AI_NO_BANNER', '1')
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field, ValidationError
from pydantic_ai import Agent
from pydantic_ai.models.function import FunctionModel, AgentInfo
from pydantic_ai.models.test import TestModel
from pydantic_ai.messages import ModelResponse, ToolCallPart, TextPart
from pydantic_ai.exceptions import UnexpectedModelBehavior
from langgraph.graph import StateGraph, END

load_dotenv(find_dotenv(usecwd=True), override=True)
print('OPENAI_API_KEY 있음 :', bool(os.getenv('OPENAI_API_KEY')))

OPENAI_API_KEY 있음 : False


## 1. 상태 (필기)

In [3]:
class AgentState(BaseModel):
    request: str = ''              # 사용자 요청
    raw_response: str = ''         # LLM 이 낸 원시 텍스트 (구조화 안 됨)
    structured_data: dict = {}     # 구조화된 결과
    error_message: str = ''        # 오류 기록

REQUEST = '현재 창고 확인 결과 고강도 볼트M8의 수량이 부족하여 경고 알림(실재고는 현재 15개 남음). 조속히 조치 요망'

## 2. 제1부 — 실패 시나리오 (교재 p.140 · 필기 그대로 실행)

LLM 대신 **'무질서한 답'** 을 정해 두고(필기의 시뮬레이션), 다음 노드가 그 글을 잘라 `int()` 로 바꾸려 한다.

In [4]:
async def unsafe_research_node(state: AgentState) -> dict:
    llm_bad_output = '''
    네, 요청하신 자재 재고 분석 결과입니다!
    ```json
    {
    'item_name': '고강도 볼트M8',
    'current_stock': '수량 부족 (재고 15개)'
    }
    ```
    추가 조치가 필요하면 말씀하세요.
    '''
    return {'raw_response': llm_bad_output}

async def unsafe_process_node(state: AgentState) -> dict:
    try:
        bad_json_str = state.raw_response.split('```json')[1].split('```')[0].strip()   # 글에서 JSON 부분 잘라내기
        data = json.loads(bad_json_str.replace("'", '"'))                              # 작은따옴표 → 큰따옴표
        stock_count = int(data['current_stock'])                                       # ← 여기서 실패
        return {'structured_data': {'item': data['item_name'], 'order_qty': 100 - stock_count}}
    except Exception as e:
        return {'error_message': f'에러 발생! 에러 내용: {e}'}

g = StateGraph(AgentState)
g.add_node('research', unsafe_research_node); g.add_node('process', unsafe_process_node)
g.set_entry_point('research'); g.add_edge('research', 'process'); g.add_edge('process', END)
unsafe_agent = g.compile()

state = await unsafe_agent.ainvoke({'request': REQUEST})
print('[결과 알림]', state['error_message'])

[결과 알림] 에러 발생! 에러 내용: invalid literal for int() with base 10: '수량 부족 (재고 15개)'


- JSON 을 잘라내고 따옴표를 고치는 '땜질' 까지는 성공했지만, **숫자 자리에 글자**(`'수량 부족 (재고 15개)'`)가 와서 `int()` 가 `ValueError` 를 냈다 (교재 p.140 의 2번)
- 📝 이런 땜질은 LLM 이 조금만 다르게 답해도(코드 블록 없이 · `json` 대신 `JSON` · 작은따옴표가 내용 안에 있음) 다른 곳에서 깨진다. 교재 p.140 의 1번 '텍스트 파싱 중 에러' 가 그것이다

## 3. 제2부 — PydanticAI `Agent` (교재 p.141~143)

`output_type` 에 BaseModel 을 주면 PydanticAI 가 ① 그 모양을 LLM 에 **도구(함수 호출) 형식**으로 알려 주고 ② 돌아온 값을 **Pydantic 으로 검사**하고 ③ 틀리면 오류 내용을 LLM 에 돌려보내 **다시 쓰게** 한다.

In [5]:
class InventoryAnalysis(BaseModel):
    item_name: str = Field(description='자재 품목 명칭')
    current_stock: int = Field(description='현재 창고에 남아있는 정확한 재고 수량 (숫자만 허용)')

SYSTEM = '당신은 제조 공장의 자재 관리 전문가입니다. 텍스트 문맥을 분석하여 재고 수량을 정확한 정수(int)로만 추출하세요.'

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

pydantic_agent = Agent(model='openai:gpt-5-nano', output_type=InventoryAnalysis, system_prompt=SYSTEM)
result = await pydantic_agent.run(f'다음 요청에서 품목 명칭과 현재 재고 수량을 정수로 추출해줘: {REQUEST}')
print(result.output, '| 요청 수 :', result.usage.requests)

In [7]:
# 키 없이 'openai:…' 모델로 Agent 를 만들면?
try:
    Agent(model='openai:gpt-5-nano', output_type=InventoryAnalysis)
except Exception as e:
    print(type(e).__name__, ':', str(e).splitlines()[0][:110])

UserError : Set the `OPENAI_API_KEY` environment variable or pass it via `OpenAIProvider(api_key=...)` to use the OpenAI p


### 🧪 가짜 모델로 '검사 → 재시도' 확인

`FunctionModel` 은 LLM 자리에 **내가 만든 함수**를 넣는다. 첫 번째 답은 일부러 필기의 실패 시나리오처럼 `current_stock='수량 부족 (재고 15개)'` 를, 두 번째 답은 `15` 를 내게 했다.

In [8]:
# 🧪 OpenAI 대신 PydanticAI 의 FunctionModel(정해 둔 규칙으로 답하는 가짜 모델) — 검사 · 재시도 · 노드 흐름은 진짜와 같다

calls = []

def careless_llm(messages, info: AgentInfo) -> ModelResponse:
    calls.append(len(messages))
    out_tool = info.output_tools[0].name                    # PydanticAI 가 만든 '답 제출용 도구' 이름
    stock = '수량 부족 (재고 15개)' if len(calls) == 1 else 15  # 첫 답은 틀리게, 두 번째는 바르게
    return ModelResponse(parts=[ToolCallPart(out_tool, {'item_name': '고강도 볼트M8', 'current_stock': stock})])

fake_agent = Agent(FunctionModel(careless_llm), output_type=InventoryAnalysis, system_prompt=SYSTEM)
result = await fake_agent.run(f'다음 요청에서 품목 명칭과 현재 재고 수량을 정수로 추출해줘: {REQUEST}')

print('최종 output :', repr(result.output), '| current_stock 자료형 :', type(result.output.current_stock).__name__)
print('LLM 호출 수 :', result.usage.requests)
print('-' * 80)
for m in result.all_messages():
    for p in m.parts:
        text = str(getattr(p, 'args', None) or getattr(p, 'content', ''))
        print(f'{type(m).__name__:<14}{type(p).__name__:<16}{text[:95]}')

최종 output : InventoryAnalysis(item_name='고강도 볼트M8', current_stock=15) | current_stock 자료형 : int
LLM 호출 수 : 2
--------------------------------------------------------------------------------
ModelRequest  SystemPromptPart당신은 제조 공장의 자재 관리 전문가입니다. 텍스트 문맥을 분석하여 재고 수량을 정확한 정수(int)로만 추출하세요.
ModelRequest  UserPromptPart  다음 요청에서 품목 명칭과 현재 재고 수량을 정수로 추출해줘: 현재 창고 확인 결과 고강도 볼트M8의 수량이 부족하여 경고 알림(실재고는 현재 15개 남음). 조속히 조치
ModelResponse ToolCallPart    {'item_name': '고강도 볼트M8', 'current_stock': '수량 부족 (재고 15개)'}
ModelRequest  ToolReturnPart  [{'type': 'int_parsing', 'loc': ['current_stock'], 'msg': 'Input should be a valid integer, una
ModelResponse ToolCallPart    {'item_name': '고강도 볼트M8', 'current_stock': 15}
ModelRequest  ToolReturnPart  Final result processed.


- 첫 답의 `current_stock` 을 Pydantic 이 검사해 **`int_parsing` 오류**를 냈고, PydanticAI 는 그 오류 목록을 `ToolReturnPart` 로 LLM 에 돌려보냈다 → 두 번째 답 `15` 가 통과 → `result.output` 은 **`InventoryAnalysis` 객체**, `current_stock` 은 **`int`**
- 그래서 다음 노드는 땜질 없이 `100 - output.current_stock` 을 계산할 수 있다 (교재 p.141 '정수 타입이 보장된 데이터')

In [9]:
# 🧪 OpenAI 대신 PydanticAI 의 FunctionModel(정해 둔 규칙으로 답하는 가짜 모델) — 검사 · 재시도 · 노드 흐름은 진짜와 같다

# 끝까지 틀린 답만 내면? — 재시도 횟수를 넘으면 예외
stubborn = Agent(FunctionModel(lambda m, info: ModelResponse(parts=[ToolCallPart(info.output_tools[0].name,
                 {'item_name': '볼트', 'current_stock': '많음'})])), output_type=InventoryAnalysis)
try:
    await stubborn.run('재고 알려줘')
except UnexpectedModelBehavior as e:
    print('UnexpectedModelBehavior :', e)

# '15' 처럼 숫자로 읽을 수 있는 글자는? — Pydantic 이 int 로 바꿔 받는다 (재시도 없음)
lenient = Agent(FunctionModel(lambda m, info: ModelResponse(parts=[ToolCallPart(info.output_tools[0].name,
                {'item_name': '볼트', 'current_stock': '15'})])), output_type=InventoryAnalysis)
r = await lenient.run('재고 알려줘')
print("'15' →", repr(r.output), '| 요청 수 :', r.usage.requests)

# TestModel — 스키마만 보고 아무 값이나 '형식에 맞게' 채워 주는 시험용 모델
print('TestModel →', repr((await Agent(TestModel(), output_type=InventoryAnalysis).run('x')).output))

UnexpectedModelBehavior : Exceeded maximum output retries (1)
'15' → InventoryAnalysis(item_name='볼트', current_stock=15) | 요청 수 : 1
TestModel → InventoryAnalysis(item_name='a', current_stock=0)


- 📝 교재 p.141~142 는 PydanticAI 가 형식을 **'강제'** · **'보장'** 한다고 쓰지만, 정확히는 **검사하고 다시 시키는 것**이다. 정리 환경(`pydantic-ai-slim 2.76.0`) 기본값으로는 재시도 1번 뒤에도 틀리면 `UnexpectedModelBehavior: Exceeded maximum output retries (1)` 이 난다 — 노드에서 `try/except` 로 받아 `error_message` 에 남기는 처리는 여전히 필요하다 (`Agent(..., retries=3)` 으로 늘릴 수 있다)
- 재시도마다 LLM 을 한 번 더 부르므로 토큰 · 시간이 든다(위 예 : 요청 2번)
- 같은 일을 LangChain 에서는 `ChatOpenAI(...).with_structured_output(InventoryAnalysis)` 로 한다 (Day 051 · 052 에서 언급) — PydanticAI 는 검사 · 재시도 · 도구 호출을 한 객체(`Agent`)로 묶은 라이브러리다
- 이름 주의 : **PydanticAI 의 `Agent`** 는 'LLM 호출 + 출력 형식 + 도구' 를 묶은 객체이고, 오늘의 LangGraph 그래프(에이전트 시스템) 안에서 **노드 하나가 부르는 부품**으로 쓴다

## 4. LangGraph 노드 안에서 쓰기 (필기 성공 시나리오)

In [10]:
# 🧪 OpenAI 대신 PydanticAI 의 FunctionModel(정해 둔 규칙으로 답하는 가짜 모델) — 검사 · 재시도 · 노드 흐름은 진짜와 같다

analysis_agent = fake_agent            # ⏳ 진짜로는 pydantic_agent (위 셀)

async def safe_research_node(state: AgentState) -> dict:
    calls.clear()                      # 가짜 모델의 '첫 답은 틀리게' 를 다시 시작
    try:
        result = await analysis_agent.run(f'다음 요청에서 품목 명칭과 현재 재고 수량을 정수로 추출해줘: {state.request}')
    except UnexpectedModelBehavior as e:                    # 📝 재시도를 넘으면 오류를 상태에 남긴다
        return {'error_message': f'구조화 실패 : {e}'}
    print(f'[Node 1] output = {result.output!r} (LLM 호출 {result.usage.requests}번)')
    return {'structured_data': result.output.model_dump()}   # BaseModel → dict

async def safe_process_node(state: AgentState) -> dict:
    if state.error_message:
        return {}
    data = state.structured_data
    order_qty = 100 - data['current_stock']                   # 땜질 없이 바로 계산
    print(f"[Node 2] 품목 : {data['item_name']} | 현재 {data['current_stock']}개 → 발주 {order_qty}개")
    return {'error_message': '없음 (비동기 파이프라인 성공)'}

g = StateGraph(AgentState)
g.add_node('safe_research', safe_research_node); g.add_node('safe_process', safe_process_node)
g.set_entry_point('safe_research'); g.add_edge('safe_research', 'safe_process'); g.add_edge('safe_process', END)
safe_agent = g.compile()

final_state = await safe_agent.ainvoke({'request': REQUEST})
print('[최종 결과(error)]', final_state['error_message'])

[Node 1] output = InventoryAnalysis(item_name='고강도 볼트M8', current_stock=15) (LLM 호출 2번)
[Node 2] 품목 : 고강도 볼트M8 | 현재 15개 → 발주 85개
[최종 결과(error)] 없음 (비동기 파이프라인 성공)


In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

async def main():
    initial_state = {"request": "현재 창고 확인 결과 고강도 볼트M8의 수량이 부족하여 경고 알림(실재고는 현재 15개 남음). 조속히 조치 요망"}
    print("에이전트 시스템 가동 ...")
    print('-'*80)
    final_state = await safe_agent.ainvoke(initial_state)
    print(f"[최종 결과(error)] {final_state['error_message']}")
await main()

에이전트 시스템 가동 ...
--------------------------------------------------------------------------------
[Node 1] PydanticAI 기반 자재 데이터 분석 중 ...
--------------------------------------------------------------------------------
요청 사항: 현재 창고 확인 결과 고강도 볼트M8의 수량이 부족하여 경고 알림(실재고는 현재 15개 남음). 조속히 조치 요망
--------------------------------------------------------------------------------
c:\Users\playdata2\miniconda3\envs\agent\Lib\site-packages\pydantic_ai\models\__init__.py:1984: DeprecatedModelWarning: Model 'openai:gpt-5-nano' is deprecated by its provider, with a retirement date of 2026-12-11. It recommends 'gpt-6-luna' instead. See https://developers.openai.com/api/docs/deprecations#2026-06-11-gpt-5-and-o3-model-deprecations
  warnings.warn(
pydantic_agent 실행 결과: 
AgentRunResult(output=InventoryAnalysis(item_name='고강도 볼트M8', current_stock=15))
--------------------------------------------------------------------------------
pydantic_agent 실행 결과: item_name='고강도 볼트M8' current_stock=15
-------------------

- 수업에서 진짜 gpt-5-nano 는 한 번에 `item_name='고강도 볼트M8' current_stock=15` 를 냈고 발주 85개로 끝났다
- 중간의 빨간 경고가 위 0절의 **gpt-5-nano 지원 종료(2026-12-11)** 예고다
- 📝 필기 상태 이름 `error_message` 에 '없음 (비동기 파이프라인 성공)' 을 넣어 성공을 알리는 것은 헷갈린다 — `status: str` 같은 항목을 따로 두는 편이 낫다 · 주석 오타 `컴스텀` → 커스텀

## 정리

| | 텍스트 파싱 (제1부) | PydanticAI `Agent` (제2부) |
|---|---|---|
| LLM 출력 | 아무 글 → 코드로 잘라 내기 | `output_type` 모양으로 함수 호출 |
| 틀린 값 | `int()` 에서 `ValueError` | Pydantic 검사 → 오류를 LLM 에 돌려줘 재시도 |
| 결과 | dict (키 · 자료형 보장 없음) | `result.output` = BaseModel 객체 (`.model_dump()` 로 dict) |
| 그래도 실패하면 | — | `UnexpectedModelBehavior` (재시도 초과) → 노드에서 처리 |